[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/08_tag_6_8_hyperparameter_gridsearch_performance.ipynb)

# Tag 6-8 - 08 Hyperparameter-Tuning und Performance

Architekturparameter beeinflussen Accuracy, Trainingszeit und Parameterzahl. Dieses Notebook testet mehrere CNN-Konfigurationen und visualisiert die Ergebnisse.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import time
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_step(model, xb, yb, loss_fn, optimizer):
    model.train()
    xb, yb = xb.to(device), yb.to(device)
    optimizer.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    return loss.item(), accuracy_from_logits(logits.detach(), yb)


def evaluate(model, loader, loss_fn=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            if loss_fn is not None:
                total_loss += loss_fn(logits, yb).item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += len(xb)
    avg_loss = total_loss / seen if loss_fn is not None else np.nan
    return avg_loss, correct / seen


def train_epochs(model, train_loader, valid_loader, loss_fn, optimizer, epochs=5):
    history = []
    for epoch in range(epochs):
        losses = []
        accs = []
        for xb, yb in train_loader:
            loss, acc = train_step(model, xb, yb, loss_fn, optimizer)
            losses.append(loss)
            accs.append(acc)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
        row = {
            "epoch": epoch + 1,
            "train_loss": float(np.mean(losses)),
            "train_acc": float(np.mean(accs)),
            "valid_loss": float(valid_loss),
            "valid_acc": float(valid_acc),
        }
        history.append(row)
        print(
            f"Epoch {row['epoch']:02d}: "
            f"train_loss={row['train_loss']:.3f}, train_acc={row['train_acc']:.3f}, "
            f"valid_loss={row['valid_loss']:.3f}, valid_acc={row['valid_acc']:.3f}"
        )
    return history


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Grid Search vorbereiten


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
y = digits.target.astype(np.int64)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
train_loader = DataLoader(TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train)), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(torch.tensor(X_valid).unsqueeze(1), torch.tensor(y_valid)), batch_size=256)

def make_cnn(width=16, kernel_size=3, pooling="max", dropout=0.0):
    padding = kernel_size // 2
    pool = nn.MaxPool2d(2) if pooling == "max" else nn.AvgPool2d(2)
    return nn.Sequential(
        nn.Conv2d(1, width, kernel_size=kernel_size, padding=padding),
        nn.ReLU(),
        pool,
        nn.Conv2d(width, width * 2, kernel_size=kernel_size, padding=padding),
        nn.ReLU(),
        pool,
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(width * 2, 10),
    )


## Mehrere Konfigurationen testen


In [ ]:
grid = []
for width in [8, 16, 32]:
    for kernel_size in [3, 5]:
        for pooling in ["max", "avg"]:
            grid.append({"width": width, "kernel_size": kernel_size, "pooling": pooling, "dropout": 0.0})
grid += [
    {"width": 16, "kernel_size": 3, "pooling": "max", "dropout": 0.25},
    {"width": 32, "kernel_size": 3, "pooling": "max", "dropout": 0.25},
]

results = []
best_valid_acc = -np.inf
best_state_dict = None
for cfg in grid:
    torch.manual_seed(RANDOM_STATE)
    model = make_cnn(**cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    start = time.time()
    history = train_epochs(model, train_loader, valid_loader, loss_fn, optimizer, epochs=5)
    seconds = time.time() - start
    result = {**cfg, "params": count_params(model), "valid_acc": history[-1]["valid_acc"], "seconds": seconds}
    results.append(result)

    if result["valid_acc"] > best_valid_acc:
        best_valid_acc = result["valid_acc"]
        best_state_dict = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

for i, row in enumerate(sorted(results, key=lambda x: x["valid_acc"], reverse=True), 1):
    print(i, row)


## Visualisieren und Optimum auswählen


In [ ]:
labels = [f"w{r['width']}_k{r['kernel_size']}_{r['pooling']}_d{r['dropout']}" for r in results]
accs = [r["valid_acc"] for r in results]
params = [r["params"] for r in results]
seconds = [r["seconds"] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(range(len(results)), accs)
axes[0].set_title("Validation Accuracy")
axes[0].set_xticks(range(len(results)))
axes[0].set_xticklabels(labels, rotation=90)
axes[1].bar(range(len(results)), params)
axes[1].set_title("Parameter")
axes[1].set_xticks(range(len(results)))
axes[1].set_xticklabels(labels, rotation=90)
axes[2].bar(range(len(results)), seconds)
axes[2].set_title("Sekunden")
axes[2].set_xticks(range(len(results)))
axes[2].set_xticklabels(labels, rotation=90)
plt.tight_layout()

best = max(results, key=lambda x: x["valid_acc"])
print("Bestes Modell nach Validierungsaccuracy:", best)


## Bestes Modell auswerten

Die Konfiguration wurde anhand der Validierungsaccuracy ausgewählt. Die folgenden Plots zeigen die Vorhersagen desselben besten Trainingslaufs auf den Validierungsdaten. Grün bedeutet korrekt, Rot bedeutet falsch.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model = make_cnn(
    width=best["width"],
    kernel_size=best["kernel_size"],
    pooling=best["pooling"],
    dropout=best["dropout"],
).to(device)
best_model.load_state_dict(best_state_dict)
best_model.eval()

with torch.no_grad():
    valid_logits = best_model(torch.tensor(X_valid).unsqueeze(1).to(device))
    valid_probabilities = torch.softmax(valid_logits, dim=1).cpu().numpy()
valid_predictions = valid_probabilities.argmax(axis=1)
valid_accuracy = (valid_predictions == y_valid).mean()
print(f"Accuracy des besten Modells auf den Validierungsdaten: {valid_accuracy:.2%}")

rng = np.random.default_rng(RANDOM_STATE)
sample_indices = rng.choice(len(X_valid), size=9, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, index in zip(axes.ravel(), sample_indices):
    true_label = y_valid[index]
    predicted_label = valid_predictions[index]
    confidence = valid_probabilities[index, predicted_label]
    color = "green" if true_label == predicted_label else "red"
    ax.imshow(X_valid[index], cmap="gray", vmin=0, vmax=1)
    ax.set_title(
        f"Wahr: {true_label} | Vorhersage: {predicted_label}\n{confidence:.0%}",
        color=color,
    )
    ax.axis("off")
fig.suptitle(f"Bestes Modell: 3x3 Vorhersagen (Accuracy: {valid_accuracy:.2%})")
plt.tight_layout()
plt.show()

ConfusionMatrixDisplay.from_predictions(
    y_valid,
    valid_predictions,
    display_labels=digits.target_names,
    cmap="Blues",
    values_format="d",
)
plt.title(f"Bestes Modell: Confusion Matrix (Accuracy: {valid_accuracy:.2%})")
plt.grid(False)
plt.show()
